<a href="https://colab.research.google.com/github/Tushar-3512/Data-Science-Practical-/blob/main/Practical_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df=pd.read_csv('/content/drive/MyDrive/TeachingRatings.csv')
df.head()

,rownames,minority,age,gender,credits,beauty,eval,division,native,tenure,students,allstudents,prof
0,1,yes,36,female,more,0.289916,4.3,upper,yes,yes,24,43,1
1,2,no,59,male,more,-0.737732,4.5,upper,yes,yes,17,20,2
2,3,no,51,male,more,-0.571984,3.7,upper,yes,yes,55,55,3
3,4,no,40,female,more,-0.677963,4.3,upper,yes,yes,40,46,4
4,5,no,31,female,more,1.509794,4.4,upper,yes,yes,42,48,5


In [6]:
print(df.columns.tolist())

['rownames', 'minority', 'age', 'gender', 'credits', 'beauty', 'eval', 'division', 'native', 'tenure', 'students', 'allstudents', 'prof']


In [7]:
print(df['gender'].unique())

['female' 'male']


In [8]:
df['female'] = (df['gender'] == 'female').astype(int)

Q1. Regression with T-test: Using the teachers rating data set, does gender
affect teaching evaluation rates?


In [9]:
model1 = smf.ols('eval ~ female', data=df).fit()
print(model1.summary())

print("""
1. Model: eval = 4.069 - 0.168*female. Male instructors average ~4.07, female ~3.90.
2. The t-test on the female coefficient gives t = -3.25, p = 0.001 (< 0.05), so we reject H0: gender has a statistically significant effect on evaluations.
3. R-squared = 0.022: gender explains only ~2% of the variation in ratings, so the effect is significant but small.
4. Regression with one 0/1 dummy is equivalent to a two-sample t-test (the coefficient is the difference in group means).
""")

                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     10.56
Date:                Thu, 19 Nov 2026   Prob (F-statistic):            0.00124
Time:                        07:26:58   Log-Likelihood:                -378.50
No. Observations:                 463   AIC:                             761.0
Df Residuals:                     461   BIC:                             769.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.0690      0.034    121.288      0.0

Q2. Regression with ANOVA: Using the teachers' rating data set, does
beauty score for instructors differ by age?


In [10]:
from statsmodels.stats.anova import anova_lm

# Create 3 age groups
df.loc[df['age'] <= 40, 'age_group'] = '40 years and younger'
df.loc[(df['age'] > 40) & (df['age'] < 57), 'age_group'] = 'between 40 and 57 years'
df.loc[df['age'] >= 57, 'age_group'] = '57 years and older'

model2 = smf.ols('beauty ~ age_group', data=df).fit()
anova_table = anova_lm(model2, typ=2)
print(anova_table)

print("""
1. ANOVA compares the mean beauty score across 3 age groups at once (a t-test only handles 2 groups).
2. F = 17.60 and p = 4.3e-08 (far below 0.05), so we reject H0: at least one age group has a different mean beauty score.
3. ANOVA only says that some group differs, not which one. A post-hoc test (e.g. Tukey HSD) is needed for that.
4. ANOVA is a regression with dummy variables: F tests whether all group coefficients are jointly zero.
""")

               sum_sq     df          F        PR(>F)
age_group   20.422743    2.0  17.597558  4.322550e-08
Residual   266.925153  460.0        NaN           NaN

1. ANOVA compares the mean beauty score across 3 age groups at once (a t-test only handles 2 groups).
2. F = 17.60 and p = 4.3e-08 (far below 0.05), so we reject H0: at least one age group has a different mean beauty score.
3. ANOVA only says that some group differs, not which one. A post-hoc test (e.g. Tukey HSD) is needed for that.
4. ANOVA is a regression with dummy variables: F tests whether all group coefficients are jointly zero.



Q3. Correlation: Using the teachers' rating dataset, Is teaching evaluation
score correlated with beauty score?

In [11]:
from scipy.stats import pearsonr

model3 = smf.ols('eval ~ beauty', data=df).fit()
print(model3.summary())

r, p = pearsonr(df['beauty'], df['eval'])
print(f"Pearson r = {r:.3f}, p-value = {p:.2e}")

print("""
1. Model: eval = 3.998 + 0.133*beauty. Each 1-unit rise in beauty score raises the predicted evaluation by ~0.13.
2. t = 4.133, p < 0.001, so the relationship is statistically significant (reject H0: slope = 0).
3. R-squared = 0.036, so r = sqrt(0.036) ≈ 0.19: a weak positive correlation. Beauty explains only ~3.6% of the variation.
4. Correlation is not causation: students may rate attractive instructors higher, but other factors may be at play.
""")

                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     17.08
Date:                Thu, 19 Nov 2026   Prob (F-statistic):           4.25e-05
Time:                        07:28:10   Log-Likelihood:                -375.32
No. Observations:                 463   AIC:                             754.6
Df Residuals:                     461   BIC:                             762.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9983      0.025    157.727      0.0